# 04 — Devoluciones y campañas (H3.2)
Pregunta: ¿dónde se concentran las devoluciones y cuánto dinero se devuelve?
Fuente: data/clean/ (salida del notebook 01).

In [1]:
import pandas as pd
from pathlib import Path

CLEAN = Path("../data/clean")
tx = pd.read_csv(CLEAN / "transacciones.csv", parse_dates=["fecha"])
pr = pd.read_csv(CLEAN / "productos.csv")
dev = pd.read_csv(CLEAN / "devoluciones.csv", parse_dates=["fecha_devolucion"])
cam = pd.read_csv(CLEAN / "campanas_marketing.csv", parse_dates=["fecha_envio"])

t = tx.merge(pr[["producto_id", "categoria", "subcategoria"]], on="producto_id")
t["devuelta"] = t.transaccion_id.isin(dev.transaccion_id)
devs = (dev.merge(tx[["transaccion_id", "producto_id", "canal"]], on="transaccion_id")
           .merge(pr[["producto_id", "categoria", "subcategoria"]], on="producto_id"))
print(f"Devoluciones: {len(dev):,} | Tasa total: {t.devuelta.mean():.2%} | "
      f"Dinero devuelto: ${dev.monto_devuelto.sum():,.0f}")

Devoluciones: 6,724 | Tasa total: 13.45% | Dinero devuelto: $2,534,466


## Tasa de devolución
Tasa = transacciones devueltas / transacciones.

In [2]:
por_categoria = t.groupby("categoria").agg(transacciones=("devuelta", "size"), devueltas=("devuelta", "sum"),
                                         tasa_pct=("devuelta", lambda s: s.mean() * 100))
dinero = devs.groupby("categoria").monto_devuelto.sum()
por_categoria["dinero_devuelto"] = dinero
por_categoria["pct_dinero"] = dinero / dinero.sum() * 100
por_categoria.sort_values("tasa_pct", ascending=False).round(1)

,transacciones,devueltas,tasa_pct,dinero_devuelto,pct_dinero
categoria,,,,,
Ropa_y_Calzado,8311,1810,21.8,320971.6,12.7
Electrónica,8419,1486,17.7,1398217.4,55.2
Deportes,8311,887,10.7,271614.9,10.7
Belleza_y_Salud,8287,878,10.6,79804.9,3.1
Alimentos_y_Bebidas,8317,845,10.2,42909.9,1.7
Hogar_y_Deco,8355,818,9.8,420947.5,16.6


In [3]:
# Por categoría y canal
(t.pivot_table(index="categoria", columns="canal", values="devuelta", aggfunc="mean") * 100).round(1)

canal,app_movil,online,tienda_fisica
categoria,,,
Alimentos_y_Bebidas,11.6,10.4,9.4
Belleza_y_Salud,10.2,11.2,10.3
Deportes,9.7,10.8,10.9
Electrónica,18.1,17.7,17.4
Hogar_y_Deco,9.6,9.8,9.8
Ropa_y_Calzado,20.9,22.6,21.4


In [4]:
# Por subcategoría, dentro de Electrónica y Ropa_y_Calzado
foco = t[t.categoria.isin(["Electrónica", "Ropa_y_Calzado"])]
(foco.groupby(["categoria", "subcategoria"]).devuelta.agg(transacciones="size", tasa_pct="mean")
     .assign(tasa_pct=lambda r: r.tasa_pct * 100).round(1))

transacciones  tasa_pct
categoria      subcategoria                            
Electrónica    Accesorios_tech           2021      16.4
               Audio                     1581      19.7
               Computadoras              2919      17.6
               Smartphones               1898      17.3
Ropa_y_Calzado Accesorios_moda           1574      22.9
               Calzado                   1264      20.9
               Ropa_femenina             2944      21.9
               Ropa_masculina            2529      21.3

In [5]:
# Motivos por categoría (% de las devoluciones de cada categoría)
print(pd.crosstab(devs.motivo, devs.categoria).loc["Talla_incorrecta"].to_dict())
(pd.crosstab(devs.motivo, devs.categoria, normalize="columns") * 100).round(1)

{'Alimentos_y_Bebidas': 0, 'Belleza_y_Salud': 0, 'Deportes': 0, 'Electrónica': 0, 'Hogar_y_Deco': 0, 'Ropa_y_Calzado': 305}


categoria,Alimentos_y_Bebidas,Belleza_y_Salud,Deportes,Electrónica,Hogar_y_Deco,Ropa_y_Calzado
motivo,,,,,,
Cambio_de_opinion,4.7,5.8,7.1,6.1,5.7,6.2
Demora_en_entrega,10.8,9.9,8.7,10.6,8.3,10.4
Error_en_pedido,13.1,14.2,14.2,13.1,13.3,12.2
Fraude_reportado,5.4,4.3,4.1,3.9,4.0,3.4
No_cumple_expectativas,39.2,37.2,36.4,38.9,40.2,22.7
Producto_defectuoso,26.7,28.5,29.5,27.5,28.4,28.3
Talla_incorrecta,0.0,0.0,0.0,0.0,0.0,16.9


In [6]:
# Electrónica por producto: ¿pocos productos o problema repartido?
elec = t[t.categoria == "Electrónica"].groupby("producto_id").devuelta.agg(n="size", tasa="mean")
elec["tasa"] *= 100
elec.tasa.describe().round(1)

count    53.0
mean     17.6
std       2.6
min      11.3
25%      16.2
50%      17.3
75%      19.3
max      24.0
Name: tasa, dtype: float64

**Hallazgos:**
- Se devolvieron 6.724 ventas (13,45%) por $2.534.466.
- **Ropa_y_Calzado tiene la tasa más alta (21,8%)**, seguida de Electrónica (17,7%). El resto está
  entre 9,8% y 10,7%. En Ropa, 16,9% de las devoluciones son por talla incorrecta (305 casos,
  0 en las demás categorías).
- **Electrónica concentra el dinero**: $1.398.217, el 55,2% de todo lo devuelto, por su ticket alto.
- **En Electrónica el problema está repartido**, no en pocos productos: los 53 productos van de
  11,3% a 24,0% con mediana de 17,3%, y las subcategorías van de 16,4% (Accesorios_tech) a 19,7% (Audio).

**H3.2 no evaluable.**
- No hay datos de costo de las campañas, así que no se puede calcular su retorno.
- La bandera `convertido` no coincide con compras reales del mismo cliente en los 30 días siguientes
  al envío (ver notebook 01, bloque 4.1, y decisión D13).